# Fine-tune YOLOv8s on Digitize-PID

This notebook validates the YOLO labels, creates the Ultralytics dataset YAML, fine-tunes yolov8s.pt, validates the best checkpoint, and renders sample predictions.

Before running it, attach either the extracted Digitize-PID dataset or the provided digitize-pid-yolo-data.zip Kaggle dataset. The notebook can extract the ZIP automatically.

In [ ]:
import sys
import subprocess
import shutil

try:
    import ultralytics  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics<9"])

import torch
from pathlib import Path
from collections import Counter
from IPython.display import display, Image
from ultralytics import YOLO

print("Ultralytics ready")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
CLASS_NAMES = [
    "Welded gate valve", "Plug valve", "Globe valve NO", "Gate valve NO",
    "Ball valve", "Butterfly valve", "Manual gate valve", "Check valve",
    "Diaphragm valve", "Needle valve", "Sealing gate valve", "Gate valve NC",
    "Globe valve NC", "Control valve", "Rotary valve NO", "Rotary valve NC",
    "Spade blind", "Spade close blind (flanged)", "Spade open blind (flanged)",
    "Right concentric reducer", "Flanged connection", "Heating coil tubes",
    "Jacketed pipe", "Mid arrow flow direction", "Circle valve",
    "Field mounted discrete indicator", "Field mounted discrete recorder",
    "Discrete with primary local access", "Discrete with auxiliary local access",
    "Solenoid actuator", "Shared with primary local access", "Shared control logic",
]

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def is_dataset_root(path: Path) -> bool:
    return all(
        (path / "images" / split).is_dir() and (path / "labels" / split).is_dir()
        for split in ("train", "val")
    )

DATASET_ROOT_OVERRIDE = None

def unpack_attached_zip() -> Path | None:
    input_root = Path("/kaggle/input")
    zip_files = sorted(
        path for path in input_root.rglob("*.zip")
        if "pid" in path.name.lower() or "digitize" in path.name.lower()
    )
    if not zip_files:
        return None
    extraction_dir = Path("/kaggle/working/attached_dataset")
    marker = extraction_dir / ".extracted"
    if not marker.exists():
        extraction_dir.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(zip_files[0]), str(extraction_dir))
        marker.write_text(zip_files[0].name, encoding="utf-8")
    return extraction_dir

def find_dataset_root() -> Path:
    if DATASET_ROOT_OVERRIDE:
        override = Path(DATASET_ROOT_OVERRIDE)
        if is_dataset_root(override):
            return override
        nested = override / "DigitizePID_Dataset"
        if is_dataset_root(nested):
            return nested
        raise FileNotFoundError(f"Invalid DATASET_ROOT_OVERRIDE: {override}")

    search_roots = [Path("/kaggle/input")]
    extracted_root = unpack_attached_zip()
    if extracted_root is not None:
        search_roots.append(extracted_root)

    for search_root in search_roots:
        candidates = [
            search_root / "digitize-pid-yolo",
            search_root / "sherifahmed-digitize-pid-yolo",
            search_root,
        ]
        for candidate in candidates:
            if is_dataset_root(candidate):
                return candidate
            nested = candidate / "DigitizePID_Dataset"
            if is_dataset_root(nested):
                return nested
        for images_train in search_root.rglob("images/train"):
            root = images_train.parent.parent
            if is_dataset_root(root):
                return root

    raise FileNotFoundError(
        "Dataset not found. Attach the extracted dataset or digitize-pid-yolo-data.zip."
    )

DATASET_ROOT = find_dataset_root()
WORK_DIR = Path("/kaggle/working/pid-yolov8s")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset root:", DATASET_ROOT)
print("Working directory:", WORK_DIR)

In [ ]:
def validate_split(split: str):
    images_dir = DATASET_ROOT / "images" / split
    labels_dir = DATASET_ROOT / "labels" / split
    images = sorted(
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    )
    labels = sorted(labels_dir.glob("*.txt"))
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    missing = sorted(image_stems - label_stems)
    orphan = sorted(label_stems - image_stems)
    counts = Counter()
    boxes = 0
    errors = []

    for label_file in labels:
        for line_no, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip():
                continue
            fields = line.split()
            if len(fields) != 5:
                errors.append(f"{label_file}:{line_no}: expected 5 fields")
                continue
            try:
                class_id = int(fields[0])
                values = [float(value) for value in fields[1:]]
            except ValueError as exc:
                errors.append(f"{label_file}:{line_no}: {exc}")
                continue
            if class_id < 0 or class_id >= len(CLASS_NAMES):
                errors.append(f"{label_file}:{line_no}: invalid class {class_id}")
            if not all(0.0 <= value <= 1.0 for value in values):
                errors.append(f"{label_file}:{line_no}: coordinates outside [0, 1]")
            if values[2] <= 0 or values[3] <= 0:
                errors.append(f"{label_file}:{line_no}: non-positive box size")
            counts[class_id] += 1
            boxes += 1

    if errors:
        raise ValueError("\n".join(errors[:20]))
    return {
        "images": len(images),
        "labels": len(labels),
        "boxes": boxes,
        "missing_labels": len(missing),
        "orphan_labels": len(orphan),
        "class_counts": counts,
    }

stats = {split: validate_split(split) for split in ("train", "val")}
for split, values in stats.items():
    print(split, values["images"], "images", values["boxes"], "boxes")
    if values["missing_labels"] or values["orphan_labels"]:
        print("  warnings:", values["missing_labels"], "missing labels;", values["orphan_labels"], "orphan labels")

combined = Counter()
for values in stats.values():
    combined.update(values["class_counts"])
print("\nClass distribution:")
for class_id, name in enumerate(CLASS_NAMES):
    print(f"{class_id:02d} {combined[class_id]:6d}  {name}")

In [ ]:
def yaml_quote(value: str) -> str:
    return '"' + value.replace("\\", "\\\\").replace('"', '\\"') + '"'

DATA_YAML = WORK_DIR / "digitize_pid_yolov8s.yaml"
yaml_lines = [
    f"path: {yaml_quote(DATASET_ROOT.resolve().as_posix())}",
    "train: images/train",
    "val: images/val",
    f"nc: {len(CLASS_NAMES)}",
    "names:",
]
yaml_lines.extend(f"  - {yaml_quote(name)}" for name in CLASS_NAMES)
DATA_YAML.write_text("\n".join(yaml_lines) + "\n", encoding="utf-8")
print(DATA_YAML.read_text())

## Fine-tune YOLOv8s

The default settings are deliberately conservative for a Kaggle T4/P100: imgsz=1280, batch=4, epochs=100, and early stopping with patience 25. If CUDA runs out of memory, reduce batch to 2 or imgsz to 1024.

In [ ]:
MODEL_PATH = "yolov8s.pt"
IMG_SIZE = 1280
BATCH_SIZE = 4
EPOCHS = 100
DEVICE = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(MODEL_PATH)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    workers=2,
    device=DEVICE,
    project=str(WORK_DIR / "runs"),
    name="digitize-pid-yolov8s",
    patience=25,
    seed=42,
    cache=False,
    pretrained=True,
    plots=True,
    exist_ok=True,
)

BEST_WEIGHTS = WORK_DIR / "runs" / "digitize-pid-yolov8s" / "weights" / "best.pt"
print("Best weights:", BEST_WEIGHTS)

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    plots=True,
)
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
val_images = sorted(
    p for p in (DATASET_ROOT / "images" / "val").iterdir()
    if p.suffix.lower() in IMAGE_SUFFIXES
)[:3]
prediction_dir = WORK_DIR / "predictions"
best_model.predict(
    source=[str(path) for path in val_images],
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(prediction_dir),
    name="val-samples",
    exist_ok=True,
)

for predicted in sorted((prediction_dir / "val-samples").glob("*")):
    if predicted.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        display(Image(filename=str(predicted), width=900))

The final artifacts are under /kaggle/working/pid-yolov8s: the dataset YAML, training plots, best.pt, last.pt, validation metrics, and prediction previews.